# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계4 : 통합 - pipeline**

## **0.미션**

단계 4에서는, 단계1,2,3 에서 생성한 함수들을 모듈화하고, 단위 테스트 및 파이프라인 코드를 작성합니다.

* **미션6**
    * 단위 테스트
        * 각 기능(함수)에 대해 단계별로 테스트를 수행하며 오류를 해결합니다.
    * 파이프라인 구축
        * 단계1의 결과가 단계2 모델에 input이 되고, 모델의 예측 결과를 기반으로
        * 응급실 추천되도록
        * 조원들이 녹음한 음성 파일에 임의의 좌표(위도, 경도)값을 부여
            * 음성파일 이름과 좌표를 저장하는 별도 데이터셋 생성
        * 각 모듈을 연결하여 파이프라인 구성하는 ipynb 파일 생성



## **1.환경설정**

### (1) 경로 설정

구글 드라이브 연결

In [1]:
path = './'

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

* requirements.txt 파일의 [경로 복사]를 한 후,
* 아래 경로에 붙여 넣기

#### 2) 라이브러리 로딩

In [35]:
#필요한 라이브러리 설치 및 불러우기
import os
import pandas as pd
import numpy as np

from haversine import haversine
import requests
import json

# 더 필요한 라이브러리 추가 -------------
import sys
sys.path.append(path)
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import openai
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import importlib
from geopy.distance import geodesic
from geopy import Point
from haversine import haversine,haversine_vector, Unit
from warnings import filterwarnings
FutureWarning
filterwarnings('ignore')

# 조에서 생성한 모듈 불러오기 -------------
import emergency
from emergency import RecommendHospital3

In [39]:
import emergency
from emergency import RecommendHospital3

importlib.reload(emergency)

<module 'emergency' from 'c:\\Users\\User\\aivle6-practice\\mini-project-7\\emergency.py'>

In [7]:
filename = "5-1.m4a"  # 예제 파일명

map_key_path = path + 'map_key.txt'

with open(map_key_path, 'r') as file:
    map_key = json.load(file)
naver_id, naver_key = map_key['c_id'], map_key['c_key']

## **2. 단위 테스트**

* 세부사항 : 아래 단계별로 데이터가 순차적으로 처리되도록 단위 테스트를 진행합니다.

In [14]:
hospital_recommender = RecommendHospital3(filename, naver_id, naver_key, path)

### (1) open ai key 등록

In [9]:
hospital_recommender.load_api_key()

### (2) audio to text to summary

In [13]:
hospital_recommender.audio_summary()

지금 편방동에 있는 집입니다. 며칠 전부터 감기 증상을 보이던 가족이 지금도 가벼운 기침과 콧물을 흘리고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶습니다. 구급차 요청드리지만 급하지 않으니 천천히 오셔도 됩니다.
 감기 증상으로 가벼운 기침과 콧물이 지속되고 있는 상황입니다. 응급상황이 아니며 약국이 문을 닫았고 병원 진료를 받고 싶어한다고 합니다. 구급차 요청은 하였지만 급한 상황은 아니니 천천히 오셔도 된다고 합니다. 해당 상황은 응급상황이 아니라 가정 간의 의료 서비스 요청으로 보입니다.


('감기 증상으로 가벼운 기침과 콧물이 지속되고 있는 상황입니다. 응급상황이 아니며 약국이 문을 닫았고 병원 진료를 받고 싶어한다고 합니다. 구급차 요청은 하였지만 급한 상황은 아니니 천천히 오셔도 된다고 합니다. 해당 상황은 응급상황이 아니라 가정 간의 의료 서비스 요청으로 보입니다.',
 np.float64(37.5384352452626),
 np.float64(126.989828026954))

### (3) 응급실 등급분류

In [15]:
hospital_recommender.classify_situation()

지금 편방동에 있는 집입니다. 며칠 전부터 감기 증상을 보이던 가족이 지금도 가벼운 기침과 콧물을 흘리고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶습니다. 구급차 요청드리지만 급하지 않으니 천천히 오셔도 됩니다.
 가벼운 기침과 콧물 증상이 있으며, 약국이 문을 닫았고 병원 진료를 받고 싶어합니다. 
응급상황은 아니지만 구급차를 요청하고 있으며 급하지 않다고 판단됩니다. 

따라서, 구급차가 요청되었지만 긴급한 상황은 아닌 것으로 보입니다.
4 등급


(4, np.float64(37.5384352452626), np.float64(126.989828026954))

### (4) 응급실추천

In [16]:
hospital_recommender.search_map()

지금 편방동에 있는 집입니다. 며칠 전부터 감기 증상을 보이던 가족이 지금도 가벼운 기침과 콧물을 흘리고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶습니다. 구급차 요청드리지만 급하지 않으니 천천히 오셔도 됩니다.
 감기 증상으로 가벼운 기침과 콧물이 지속 중이며, 병원 진료를 받고 싶어합니다. 
응급상황은 아니지만 약국이 문을 닫아 구급차를 요청하고 있습니다. 
현재 상황은 급하지 않으니 천천히 오셔도 된다고 합니다.
4 등급


'가까운 병원을 찾아가는 것을 추천드립니다.'

## **3. 파이프라인**

* 세부사항
    * [2. 단계별 테스트] 의 내용을 순차적으로 정리합니다.
        * 데이터 처리 전 준비작업 : 한번 실행하면 되는 영역
            * 키, 데이터로딩
            * 모델/토크나이저 로딩
        * 입력값이 들어 왔을 때 출력값까지 처리되는 영역

In [40]:
filename = "1-1.m4a"  # 예제 파일명

map_key_path = path + 'map_key.txt'

with open(map_key_path, 'r') as file:
    map_key = json.load(file)
naver_id, naver_key = map_key['c_id'], map_key['c_key']

In [41]:
hospital_recommender = RecommendHospital3(filename, naver_id, naver_key, path)
result = hospital_recommender.search_map()
print(result)

신동현대아파트입니다. 제 아버지가 갑자기 가슴을 움켜주더니 의식을 잃고 바닥에 쓰러졌습니다. 호흡이 멈춘 상태고 제가 지금 심폐소생술을 시도하고 있습니다. 세세동기같은 응급장비가 필요할 것 같습니다. 빨리 구급차 보내주세요. 의식이 돌아오지 않고 있습니다.
 가슴을 움켜주다가 의식을 잃고 바닥에 쓰러지셨고, 호흡이 멈춘 상태라고 합니다. 현재 심폐소생술을 시도 중이고, 세세동기 같은 응급장비가 필요할 것으로 판단됩니다. 신고자가 빨리 구급차를 요청하고 있으며, 환자의 의식이 돌아오지 않고 있는 것으로 보입니다. 이는 심각한 응급상황으로 판단됩니다.
1 등급
{'목적지': ['분당서울대학교병원', '대진의료재단분당제생병원', '국군수도병원', '차의과학대학교분당차병원', '성모윌병원'], '경과시간': ['0시간 9분', '0시간 11분', '0시간 15분', '0시간 19분', '0시간 22분'], '거리': [2.626, 4.866, 5.973, 7.666, 10.909], '택시비+톨비': [5800, 7300, 8200, 10300, 12800]}


In [24]:
pd.DataFrame(result)

,목적지,경과시간,거리,택시비+톨비
0,분당서울대학교병원,0시간 9분,2.626,6000
1,대진의료재단분당제생병원,0시간 13분,4.707,7300
2,국군수도병원,0시간 14분,5.973,8200
3,차의과학대학교분당차병원,0시간 17분,7.443,10000
4,성모윌병원,0시간 21분,10.819,12600
